In [ ]:
# Uninstall the currently installed version of the Transformers library.
# This helps avoid version conflicts with the version we want to install.
!pip uninstall -y transformers

# Install specific versions of the required libraries.
!pip install -U transformers==4.46.3 \
                accelerate \
                sentencepiece \
                huggingface-hub==0.26.2

In [ ]:
# Import the pytorch library for tensor operations and deep learning
import torch
# Import Model and tokenizer and pieline from transformers library
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# Load model(Phi-3) and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
model = AutoModelForCausalLM.from_pretrained(
"microsoft/Phi-3-mini-4k-instruct",
# Automatically selects the available device CPU or GPU
device_map="auto",
#Automatically select  the appropriate  data type
torch_dtype="auto",
# Allow the model to run its own code
trust_remote_code=
True
,
)
# Create a pipeline
generator = pipeline(
"text-generation", # tells the pipeline that the model to generate text
model=model, # previously leaded model
tokenizer=tokenizer, # previously loaded tokenizer
# return the only generated text not the original prompt
return_full_text=
False
,
# Generates a maximum of 50 new tokens
max_new_tokens=50,
# Disable random sampling ,so the model can mostly generate the next coming token not random
do_sample=
False
,
)

In [ ]:
# Tell the model to perform a task using a variable
# This prompt tell the model what to generate
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
# generates the output using the text generation pipeline
output = generator(prompt)
#output[0] accesses the first generated result.
#'generated_text' retrieves the actual generated text.
print(output[0]['generated_text'])

In [ ]:
prompt = "The capital of France is"
# Tokenize the input prompt
# tokenizer convert the text into token Ids
# return_tensors="pt" returns the tokens as PyTorch tensors.
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
# Move the token IDs to CPU
input_ids = input_ids.to("cpu")
# Pass the token IDs through the Transformer model.The model.model() function runs only the Transformer layers.
# This returns the hidden representations (embeddings) from the last Transformer layer.
# The LM Head is NOT applied yet.
model_output = model.model(input_ids)
#lm_head =LM head is the final layer in the transformer block .It job is to take the hidden representaations .and generate scores
# model_output= contain the hidden representations (embeddings)
# model.lm_head it predicts the next token
lm_head_output = model.lm_head(model_output[0])

In [ ]:
#select the scores of first and last input  token
# Find the token ID with the highest  relevance score using argmax().
# argmax(-1) it returns the index of tokens
token_id = lm_head_output[0,-1].argmax(-1)
# tokenizer write the tokens back into text
tokenizer.decode(token_id)

In [ ]:
#Shows the shape
model_output[0].shape

In [ ]:
# Shows the shape
lm_head_output.shape

In [ ]:
# Prompt tells the model to perform a task based in the prompt
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# Tokenizer convert the text into token IDs
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
# Move the token IDs to CPU
input_ids = input_ids.to("cpu")


In [ ]:
# Measure time how long it can take to execute the code
%%timeit -n 1
# Generate the text using the model
generation_output = model.generate(
# generated token IDs
input_ids=input_ids,
# Generate max 100 new tokens
max_new_tokens=100,
#True = Model stores the previous calculations  and reuse them
use_cache=
True
)

In [ ]:
#Measure time how long it can take  to execute the code once
%%timeit -n 1
# Generate the text using the mdoel
generation_output = model.generate(
    # Previously generated tokens
input_ids=input_ids,
# Generate the max 100 new tokens
max_new_tokens=100,
# False =Model recalculates everything from the begining everytime it generate the tokens
use_cache=
False
)